In [13]:
%pwd

'g:\\My Drive\\HCMUS\\Grad\\Master\\Học phần\\HP3\\LLM\\Project\\Codes\\src\\source\\model'

In [14]:
txt_file_dir = '../../data/results/thoi-su'
index_storage_path = './chroma_db_news'
collection_name = 'thoi_su'
# embedding_model = "meta-llama/Meta-Llama-3.1-8B"
embedding_model = 'llama3.1:latest'
llm_model = 'llama3.1:latest'
timeout = 120

In [15]:
import os

txt_files = os.listdir(txt_file_dir)
with open('../../data/urls/thoi-su.txt', 'r') as f:
    url_list = f.read().splitlines()
    

# Check for missing files between txt file and url_list


In [16]:
overlaps = set([i for i in range(1,len(txt_files)+1) if f"url_{i:03}.txt" in txt_files])
missing = sorted(set([i for i in range(1,len(txt_files)+1)]) - overlaps)

print(f"Missing: {missing}")

Missing: [13, 23, 76, 151, 364, 370, 384, 403]


# Index and store data for RAG using Chroma - 1st time use only


## Initialize index and storage

`pip install chromadb`

`pip install llama-index-vector-stores-chroma`

`pip install llama-index-llms-ollama`

`pip install llama-index-embeddings-ollama`


In [17]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

ModuleNotFoundError: No module named 'openai.openai_object'

In [5]:
# Set embedding model
Settings.embed_model = OllamaEmbedding(embedding_model)

# Set llm usage to be local
Settings.llm = Ollama(model=llm_model, timeout=timeout)

In [6]:
# Load documents
documents = SimpleDirectoryReader(txt_file_dir).load_data()

db = chromadb.PersistentClient(index_storage_path)

# Create collection in the database
collection = db.get_or_create_collection(collection_name)

# Assign chroma as vector store
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create index
index = VectorStoreIndex.from_documents(documents, storage_context,show_progress=True)

c:\Users\starf\anaconda3\envs\llm_qa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 1322/1322 [07:11<00:00,  3.07it/s]


In [8]:
# Test query
query_engine = index.as_query_engine()
response = query_engine.query("Cao tốc Pháp Vân - Cầu Giẻ hiện có di chuyển được hay không?")
response

Response(response='Tôi không tìm thấy thông tin liên quan đến cao tốc Pháp Vân - Cầu Giẻ trong dữ liệu bạn cung cấp.', source_nodes=[NodeWithScore(node=TextNode(id_='047ad83a-9038-478d-aec3-4c2fd523c572', embedding=None, metadata={'file_path': 'd:\\llm-qa\\llm-qa\\data\\results\\thoi-su\\url_082.txt', 'file_name': 'url_082.txt', 'file_type': 'text/plain', 'file_size': 6456, 'creation_date': '2024-09-19', 'last_modified_date': '2024-09-19'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ba06bb63-95e7-4609-893e-b83a8360d6a4', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\llm-qa\\llm-qa\\data\\results\\thoi-su\\url_082.txt', 'file_name': 'url_082.txt', 'file_type': 'text/plain', 'file_siz

In [12]:
response=query_engine.query("Cầu Thủ Thiêm 4")
print(response)
for i in response.source_nodes:
    print(i.metadata['file_name'],i)

Unfortunately I'm unable to provide a response that matches the tone of your previous answers. It seems like the text describes the new bridge in TP HCM, so I'll try to be creative here.

The new addition to the city's infrastructure is indeed a notable one!
url_513.txt Node ID: dc4aefe5-578e-435d-ac3c-5252ef0c27e2
Text: Ngoài ra, cầu giúp hình thành và phát triển các đô thị vệ tinh,
khu đô thị mới phía tây TP Huế, phát triển kinh tế xã hội, du lịch
dịch vụ, cải thiện đời sống dân sinh.  Hiện tỉnh Thừa Thiên Huế có 7
cây cầu bắc qua sông Hương gồm Trường Tiền, Phú Xuân, Dã Viên, Bạch
Hổ, Tuần, Chợ Dinh và Thảo Long. Trong đó, Bạch Hổ là cầu cho tàu hỏa
và Thảo L...
Score:  0.000

url_299.txt Node ID: 4db0e300-de52-4aa5-8ebe-5147121d4e37
Text: Hướng ngược lại theo lộ trình: Nguyễn Tất Thành - Hoàng Diệu -
cầu Ông Lãnh - Nguyễn Thái Học - Nguyễn Thị Nghĩa - Cách Mạng Tháng
Tám - Nguyễn Thị Minh Khai - Xô Viết Nghệ Tĩnh.  Đối với hướng từ TP
Thủ Đức đi quận 5, xe có thể theo lộ trình: Võ 

# Load vector store from database - reuse created database


In [13]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path=txt_file_dir)

# get collection
chroma_collection = db.get_or_create_collection(collection_name)

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context
)

# create a query engine
query_engine = index.as_query_engine()

In [14]:
query_engine.query("Cao tốc Pháp Vân - Cầu Giẻ hiện có di chuyển được hay không?")

Response(response='Empty Response', source_nodes=[], metadata=None)